# 04 - Yield Forecasting (5 km grid + ML regression)

Extracts gridded predictors (VHI stats, NDVI integral, SAR phenology, IMD rainfall/temperature proxies), trains **RandomForest** and **GradientBoosting** regressors against district yield history, and aggregates to district/state forecasts.

> `data/sample/district_yield_history.csv` is **synthetic** - swap in official Ministry crop-cutting statistics with the same schema.

In [ ]:
import sys, yaml
sys.path.append('..')
import numpy as np, pandas as pd
from src import gee_utils, indices, yield_model, visualization

with open('../config/config.yaml') as f:
    cfg = yaml.safe_load(f)
YCFG = cfg['yield_model']
hist = pd.read_csv('../data/sample/district_yield_history.csv')
hist.head()

## Predictor features per district-season
Operational path: GEE regional reductions of VHI / NDVI integral / S1 metrics over each district x year. A reproducible **offline fallback** generates physically-plausible features so the ML chain runs without GEE access.

In [ ]:
USE_GEE = False  # set True to extract real features (slow; needs district boundaries)

if USE_GEE:
    import ee
    gee_utils.init_ee()
    # Example for one district-season: reduce VHI/NDVI over GAUL level-2 geometry
    # dist_g = ee.FeatureCollection('FAO/GAUL/2015/level2') \
    #     .filter(ee.Filter.eq('ADM2_NAME', 'Ludhiana')).geometry()
    # ... loop districts x years, fill feats_df
    raise NotImplementedError('Wire district geometries, then loop extraction')
else:
    rng = np.random.default_rng(42)
    feats = hist.copy()
    # Plausible couplings: better VHI / rainfall -> higher yield
    feats['vhi_mean'] = 35 + 12 * (feats.yield_t_ha / feats.yield_t_ha.max()) + rng.normal(0, 2, len(feats))
    feats['ndvi_integral'] = 60 + 25 * (feats.yield_t_ha / feats.yield_t_ha.max()) + rng.normal(0, 3, len(feats))
    feats['vh_amplitude_db'] = 4 + 3 * (feats.yield_t_ha / feats.yield_t_ha.max()) + rng.normal(0, 0.5, len(feats))
    feats['rain_mm'] = 80 + 60 * (feats.yield_t_ha / feats.yield_t_ha.max()) + rng.normal(0, 10, len(feats))
    feats['heat_stress_days'] = np.maximum(0, 12 - 8 * (feats.yield_t_ha / feats.yield_t_ha.max()) + rng.normal(0, 2, len(feats)))
FEATURES = ['vhi_mean', 'ndvi_integral', 'vh_amplitude_db', 'rain_mm', 'heat_stress_days']
feats.head()

In [ ]:
# Train on past years, forecast the latest year
test_year = feats.year.max()
train = feats[feats.year < test_year]
test = feats[feats.year == test_year]
X_tr, y_tr = train[FEATURES], train['yield_t_ha']
X_te, y_te = test[FEATURES], test['yield_t_ha']

models = yield_model.build_models(YCFG)
cv = yield_model.cross_validate(models, X_tr, y_tr, YCFG['cv_folds'])
cv

In [ ]:
best_name = cv.sort_values('r2_mean', ascending=False).iloc[0]['model']
best = models[best_name]
pred = yield_model.fit_and_predict(best, X_tr, y_tr, X_te)
metrics = yield_model.evaluate(y_te, pred)
print('Best model:', best_name, '|', metrics)

forecast = test[['state', 'district']].copy()
forecast['yield_pred'] = pred
forecast['yield_actual'] = y_te.values
forecast.to_csv('../outputs/yield_forecast.csv', index=False)
forecast

In [ ]:
visualization.yield_forecast_bars(forecast)

In [ ]:
# State-level aggregation (sown-area weighted)
w = test[['state', 'district', 'sown_area_kha']]
state_fc = (forecast.merge(w, on=['state', 'district'])
            .groupby('state')
            .apply(lambda d: np.average(d.yield_pred, weights=d.sown_area_kha), include_groups=False)
            .rename('state_yield_t_ha').reset_index())
state_fc.to_csv('../outputs/state_yield_forecast.csv', index=False)
state_fc

## Feature importance
Sanity-check that vegetation health and rainfall dominate, and heat-stress days act negatively.

In [ ]:
import matplotlib.pyplot as plt
imp = pd.Series(best.feature_importances_, index=FEATURES).sort_values()
imp.plot.barh(color='teal', figsize=(7, 3), title=f'Feature importance ({best_name})')
plt.tight_layout()